In [100]:
import pandas as pd 
import numpy as np 

o_vev = pd.read_csv('voteevent.csv')
o_bvev = pd.read_csv('billvoteevent.csv')

In [101]:
vev = o_vev.iloc[1066:1334].reset_index(drop=True)
vev['votes']

0      [{"voter_name_raw":"กฤษฎา ตันเทอดทิตย์","voter...
1      [{"voter_name_raw":"กฤษฎา ตันเทอดทิตย์","voter...
2      [{"voter_name_raw":"กนก ลิ้มตระกูล","voter_par...
3      [{"voter_name_raw":"กนก ลิ้มตระกูล","voter_par...
4      [{"voter_name_raw":"กนก ลิ้มตระกูล","voter_par...
                             ...                        
263    [{"voter_name_raw":"ธิษะณา ชุณหะวัณ","voter_pa...
264    [{"voter_name_raw":"นิตยา มีศรี","voter_party"...
265    [{"voter_name_raw":"ปรีติ เจริญศิลป์","voter_p...
266    [{"voter_name_raw":"พงศธร ศรเพชรนรินทร์","vote...
267    [{"voter_name_raw":"ธัญธร ธนินวัฒนาธร","voter_...
Name: votes, Length: 268, dtype: object

In [102]:
bvev = o_bvev.iloc[0:631].copy()
bvev.loc[bvev["vote_events"] == "[]", "vote_events"] = pd.NA
bvev = bvev.dropna(subset=["vote_events"]).reset_index(drop=True)
bvev['vote_events']

0      [{"abstain_count":1,"agree_count":249,"novote_...
1      [{"abstain_count":1,"agree_count":249,"novote_...
2      [{"abstain_count":3,"agree_count":262,"novote_...
3      [{"abstain_count":7,"agree_count":259,"novote_...
4      [{"abstain_count":1,"agree_count":249,"novote_...
                             ...                        
152    [{"abstain_count":2,"agree_count":269,"novote_...
153    [{"abstain_count":4,"agree_count":257,"novote_...
154    [{"abstain_count":15,"agree_count":298,"novote...
155    [{"abstain_count":1,"agree_count":316,"novote_...
156    [{"abstain_count":11,"agree_count":236,"novote...
Name: vote_events, Length: 157, dtype: object

In [73]:
total_event = vev.shape[0] + bvev.shape[0]
total_event

425

In [114]:
import json
import re
from difflib import SequenceMatcher

ABSENT_OPTION = "ลา / ขาดลงมติ"

def parse_json_array(raw):
    if pd.isna(raw):
        return []
    if isinstance(raw, list):
        return raw

    text = str(raw).strip()
    if text == "" or text == "[]":
        return []

    try:
        parsed = json.loads(text)
    except Exception:
        # CSV-escaped JSON often stores double quotes as ""
        parsed = json.loads(text.replace('""', '"'))

    return parsed if isinstance(parsed, list) else []

def normalize_votes_table(votes_df):
    if votes_df.empty:
        return pd.DataFrame(columns=["voter_name_raw", "voter_party", "option"])

    out = votes_df.reindex(columns=["voter_name_raw", "voter_party", "option"]).copy()
    out["voter_name_raw"] = out["voter_name_raw"].astype(str).str.strip()
    out["voter_party"] = out["voter_party"].astype(str).str.strip()
    out["option"] = out["option"].astype(str).str.strip()
    out = out[
        (out["voter_party"] != "") &
        (out["voter_party"] != "-") &
        (out["voter_party"].str.lower() != "nan")
    ].copy()
    return out

def extract_votes_from_vote_events(raw):
    events = parse_json_array(raw)
    votes = []
    for event in events:
        if isinstance(event, dict):
            event_votes = event.get("votes", [])
            if isinstance(event_votes, list):
                votes.extend(event_votes)
    return votes

# Source 1: vev["votes"]
vev_votes_long = (
    vev["votes"]
    .apply(parse_json_array)
    .explode()
    .dropna()
    .apply(pd.Series)
    .pipe(normalize_votes_table)
)

# Source 2: bvev["vote_events"] -> each event["votes"]
bvev_votes_long = (
    bvev["vote_events"]
    .apply(extract_votes_from_vote_events)
    .explode()
    .dropna()
    .apply(pd.Series)
    .pipe(normalize_votes_table)
)

# Merge both sources
all_votes_long = pd.concat([vev_votes_long, bvev_votes_long], ignore_index=True)

# -------------------------------------------------------------------
# 1) Party-name normalization for OCR noise
# -------------------------------------------------------------------
CANONICAL_PARTIES = [
    "สมาชิกวุฒิสภา",
    "เพื่อไทย",
    "ก้าวไกล",
    "พลังประชารัฐ",
    "ภูมิใจไทย",
    "ประชาธิปัตย์",
    "รวมไทยสร้างชาติ",
    "ไทยสร้างไทย",
    "ชาติไทยพัฒนา",
    "ประชาชาติ",
    "เสรีรวมไทย",
    "เพื่อชาติ",
    "เศรษฐกิจใหม่",
    "พลังปวงชนไทย",
    "ประชาภิวัฒน์",
    "ไทยศรีวิไลย์",
    "พลังท้องถิ่นไท",
    "ชาติพัฒนา",
    "เศรษฐกิจไทย",
    "กล้าธรรม",
    "รวมพลัง",
    "พลังชาติไทย",
    "พลังธรรมใหม่",
    "พลเมืองไทย",
    "ประชาธิปไตยใหม่",
    "ไทรักธรรม",
    "ครูไทยเพื่อประชาชน",
    "รักษ์ผืนป่าประเทศไทย",
    "พรรคโอกาสไทย",
    "รวมแผ่นดิน",
    "พลังไทยรักไทย",
    "อำนาจใหม่",
    "อนาคตใหม่",
    "ประชานิยม",
]

REGEX_CANONICAL_RULES = [
    (r"ว.?ฒิสภ", "สมาชิกวุฒิสภา"),
    (r"รวมไทยสร้างชาติ", "รวมไทยสร้างชาติ"),
    (r"ไทยสร้างไทย", "ไทยสร้างไทย"),
    (r"พลังประชารัฐ", "พลังประชารัฐ"),
    (r"ภูมิใจไทย", "ภูมิใจไทย"),
    (r"ประชาธิป", "ประชาธิปัตย์"),
    (r"ก้าวไกล", "ก้าวไกล"),
    (r"เพื่อไทย", "เพื่อไทย"),
    (r"ชาติไทยพัฒนา|คาติไทยพัฒนา|ชาดิไทยพัฒนา", "ชาติไทยพัฒนา"),
    (r"ประชาชาติ", "ประชาชาติ"),
    (r"เสรีรวมไทย", "เสรีรวมไทย"),
    (r"เศรษฐกิจใหม่", "เศรษฐกิจใหม่"),
    (r"เพื่อชาติ", "เพื่อชาติ"),
    (r"พลังปวงชนไทย", "พลังปวงชนไทย"),
    (r"ประชาภิวัฒน์", "ประชาภิวัฒน์"),
    (r"ไทยศรีวิไลย์", "ไทยศรีวิไลย์"),
    (r"พลังท้องถิ่นไท", "พลังท้องถิ่นไท"),
    (r"ชาติพัฒนา", "ชาติพัฒนา"),
    (r"เศรษฐกิจไทย", "เศรษฐกิจไทย"),
    (r"กล้าธรรม|กลาธรรม", "กล้าธรรม"),
    (r"รวมพลัง", "รวมพลัง"),
    (r"พลังชาติไทย", "พลังชาติไทย"),
    (r"พลังธรรมใหม่", "พลังธรรมใหม่"),
    (r"พลเมืองไทย", "พลเมืองไทย"),
    (r"ประชาธิปไตยใหม่", "ประชาธิปไตยใหม่"),
    (r"ไทรักธรรม", "ไทรักธรรม"),
    (r"ครูไทยเพื่อประชาชน", "ครูไทยเพื่อประชาชน"),
    (r"รักษ์ผืนป่าประเทศไทย", "รักษ์ผืนป่าประเทศไทย"),
    (r"พรรคโอกาสไทย", "พรรคโอกาสไทย"),
    (r"รวมแผ่นดิน", "รวมแผ่นดิน"),
    (r"พลังไทยรักไทย", "พลังไทยรักไทย"),
    (r"อำนาจใหม่", "อำนาจใหม่"),
    (r"อนาคตใหม่", "อนาคตใหม่"),
    (r"ประชานิยม", "ประชานิยม"),
]

def basic_party_clean(text):
    text = str(text).strip()
    text = text.replace("|", " ")
    text = text.replace("`", "")
    text = text.replace("'", "")
    text = text.replace('"', "")
    text = re.sub(r"[A-Za-z0-9_]+", "", text)
    text = re.sub(r"[^\u0E00-\u0E7F\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"^พรรค", "", text).strip()
    return text

def map_party_by_rules(cleaned_text):
    for pattern, canonical in REGEX_CANONICAL_RULES:
        if re.search(pattern, cleaned_text):
            return canonical
    return cleaned_text

def fuzzy_match_party(text, choices, threshold=0.84):
    if len(text) < 4:
        return text

    best_choice = text
    best_score = 0.0
    for choice in choices:
        score = SequenceMatcher(None, text, choice).ratio()
        if score > best_score:
            best_score = score
            best_choice = choice

    return best_choice if best_score >= threshold else text

def normalize_party_name(raw_name, choices):
    cleaned = basic_party_clean(raw_name)
    if cleaned == "":
        return ""
    rule_mapped = map_party_by_rules(cleaned)
    return fuzzy_match_party(rule_mapped, choices)

raw_parties = all_votes_long["voter_party"].dropna().astype(str)
raw_unique_parties = sorted(raw_parties.unique())
party_choices = sorted(set(CANONICAL_PARTIES))
party_name_map = {raw: normalize_party_name(raw, party_choices) for raw in raw_unique_parties}

all_votes_long["voter_party_norm"] = all_votes_long["voter_party"].map(party_name_map)
all_votes_long = all_votes_long[
    (all_votes_long["voter_party_norm"] != "") &
    (all_votes_long["voter_party_norm"] != "-")
]

# -------------------------------------------------------------------
# 2) Person-name normalization for OCR noise (within each party)
# -------------------------------------------------------------------
MANUAL_NAME_OVERRIDES = {
    # Example: ("เพื่อไทย", "ชอจาก OCR") : "ชื่อที่ถูกต้อง",
}

def basic_name_clean(text):
    text = str(text).strip()
    text = text.replace("|", " ")
    text = text.replace("`", "")
    text = text.replace("'", "")
    text = text.replace('"', "")
    text = re.sub(r"[A-Za-z0-9_]+", "", text)
    text = re.sub(r"[^\u0E00-\u0E7F\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def names_similar(name_a, name_b, threshold=0.94):
    a_key = re.sub(r"\s+", "", name_a)
    b_key = re.sub(r"\s+", "", name_b)

    if len(a_key) < 8 or len(b_key) < 8:
        return False
    if abs(len(a_key) - len(b_key)) > 3:
        return False

    return SequenceMatcher(None, a_key, b_key).ratio() >= threshold

all_votes_long["voter_name_clean"] = all_votes_long["voter_name_raw"].apply(basic_name_clean)
all_votes_long = all_votes_long[all_votes_long["voter_name_clean"] != ""]

name_freq = (
    all_votes_long.groupby(["voter_party_norm", "voter_name_clean"], as_index=False)
    .size()
    .rename(columns={"size": "freq"})
)

name_map = {}
for party, group_df in name_freq.groupby("voter_party_norm", sort=False):
    canonical_names = []
    ordered = group_df.sort_values(["freq", "voter_name_clean"], ascending=[False, True])

    for _, row in ordered.iterrows():
        candidate = row["voter_name_clean"]
        matched = None
        for canonical in canonical_names:
            if names_similar(candidate, canonical):
                matched = canonical
                break

        if matched is None:
            canonical_names.append(candidate)
            matched = candidate

        name_map[(party, candidate)] = matched

def normalize_name(party, cleaned_name):
    manual_key = (party, cleaned_name)
    if manual_key in MANUAL_NAME_OVERRIDES:
        return MANUAL_NAME_OVERRIDES[manual_key]
    return name_map.get((party, cleaned_name), cleaned_name)

all_votes_long["voter_name_norm"] = [
    normalize_name(party, cleaned_name)
    for party, cleaned_name in zip(all_votes_long["voter_party_norm"], all_votes_long["voter_name_clean"])
]

# Mark absent/present
all_votes_long["absent"] = (all_votes_long["option"] == ABSENT_OPTION).astype(int)
all_votes_long["present"] = (all_votes_long["option"] != ABSENT_OPTION).astype(int)

# Person-level absence rate = absent / (absent + present)
person_absence = (
    all_votes_long
    .groupby(["voter_party_norm", "voter_name_norm"], as_index=False)[["absent", "present"]]
    .sum()
)
person_absence["absence_rate"] = (
    person_absence["absent"] / (person_absence["absent"] + person_absence["present"])
)

# Party-level average of person absence rates
party_average_absence = (
    person_absence.groupby("voter_party_norm", as_index=False)
    .agg(
        people=("voter_name_norm", "nunique"),
        avg_absence_rate=("absence_rate", "mean")
    )
    .rename(columns={"voter_party_norm": "voter_party"})
    .sort_values("avg_absence_rate", ascending=False)
)
party_average_absence["avg_absence_pct"] = (party_average_absence["avg_absence_rate"] * 100).round(2)

# QA tables
party_mapping_review = (
    pd.DataFrame({
        "voter_party_raw": list(party_name_map.keys()),
        "voter_party_norm": list(party_name_map.values()),
    })
    .sort_values(["voter_party_norm", "voter_party_raw"])
)

name_mapping_review = (
    all_votes_long[["voter_party_norm", "voter_name_raw", "voter_name_clean", "voter_name_norm"]]
    .drop_duplicates()
    .sort_values(["voter_party_norm", "voter_name_norm", "voter_name_clean"])
)

party_average_absence

,voter_party,people,avg_absence_rate,avg_absence_pct
34,โอกาสไทย,2,0.833333,83.33
38,ไทยศรีวิไลย์,1,0.745946,74.59
23,รวมแผ่นดิน,1,0.600000,60.00
28,เป็นธรรม,2,0.523411,52.34
29,เพื่อชาติ,11,0.492675,49.27
36,ไทยก้าวหน้า,1,0.491803,49.18
33,เสรีรวมไทย,17,0.483652,48.37
13,พลังชาติไทย,2,0.437127,43.71
11,ประชาภิวัฒน์,2,0.435543,43.55
24,รวมไทยสร้างชาติ,42,0.427282,42.73


In [121]:
party_average_absence.to_csv('vote_absence.csv')

In [118]:
# Manual fix for remaining OCR name pairs
MANUAL_NAME_GLOBAL_OVERRIDES = {
    "ไพโรจน์ พานิขสมัย": "ไพโรจน์ พานิชสมัย",
    "กนิษฐ์ ขาญปรีชญา": "กนิษฐ์ ชาญปรีชญา",
}

def _norm_space(text):
    return re.sub(r"\s+", " ", str(text)).strip()

override_map = {
    _norm_space(k): _norm_space(v)
    for k, v in MANUAL_NAME_GLOBAL_OVERRIDES.items()
}

all_votes_long["voter_name_norm"] = all_votes_long["voter_name_norm"].apply(
    lambda x: override_map.get(_norm_space(x), _norm_space(x))
)
all_votes_long["voter_name_clean"] = all_votes_long["voter_name_clean"].apply(
    lambda x: override_map.get(_norm_space(x), _norm_space(x))
)

# Recompute summaries after manual name overrides
person_absence = (
    all_votes_long
    .groupby(["voter_party_norm", "voter_name_norm"], as_index=False)[["absent", "present"]]
    .sum()
)
person_absence["absence_rate"] = (
    person_absence["absent"] / (person_absence["absent"] + person_absence["present"])
)

party_average_absence = (
    person_absence.groupby("voter_party_norm", as_index=False)
    .agg(
        people=("voter_name_norm", "nunique"),
        avg_absence_rate=("absence_rate", "mean")
    )
    .rename(columns={"voter_party_norm": "voter_party"})
    .sort_values("avg_absence_rate", ascending=False)
)
party_average_absence["avg_absence_pct"] = (party_average_absence["avg_absence_rate"] * 100).round(2)

name_mapping_review = (
    all_votes_long[["voter_party_norm", "voter_name_raw", "voter_name_clean", "voter_name_norm"]]
    .drop_duplicates()
    .sort_values(["voter_party_norm", "voter_name_norm", "voter_name_clean"])
)

# Quick check for the reported pairs
target_names = [
    "ไพโรจน์ พานิขสมัย",
    "ไพโรจน์ พานิชสมัย",
    "กนิษฐ์ ขาญปรีชญา",
    "กนิษฐ์ ชาญปรีชญา",
]

name_mapping_review[
    name_mapping_review["voter_name_clean"].isin(target_names) |
    name_mapping_review["voter_name_norm"].isin(target_names)
]

,voter_party_norm,voter_name_raw,voter_name_clean,voter_name_norm
248,สมาชิกวุฒิสภา,กนิษฐ์ ชาญปรีชญา,กนิษฐ์ ชาญปรีชญา,กนิษฐ์ ชาญปรีชญา
3124,สมาชิกวุฒิสภา,กนิษฐ์ ขาญปรีชญา,กนิษฐ์ ชาญปรีชญา,กนิษฐ์ ชาญปรีชญา
4562,สมาชิกวุฒิสภา,กนิษฐ์-ชาญปรีชญา,กนิษฐ์ชาญปรีชญา,กนิษฐ์ ชาญปรีชญา
137,สมาชิกวุฒิสภา,ไพโรจน์ พานิชสมัย,ไพโรจน์ พานิชสมัย,ไพโรจน์ พานิชสมัย
10852,สมาชิกวุฒิสภา,ไพโรจน์ พานิขสมัย,ไพโรจน์ พานิชสมัย,ไพโรจน์ พานิชสมัย
4460,สมาชิกวุฒิสภา,ไพโรจน์-พานิชสมัย,ไพโรจน์พานิชสมัย,ไพโรจน์ พานิชสมัย


In [122]:
# Reference-assisted cleanup (web source): official Senate name lists
SENATE_REFERENCE_CSV_URLS = [
    "https://datacatalog.senate.go.th/dataset/sec_03_1101/resource/2abbb3f8-9d7e-436b-b026-07fe5e5f1e9a/download",
    "https://datacatalog.senate.go.th/dataset/sec_03_1101/resource/b1211e46-763c-4ad2-a461-e342b9e2a933/download",
]
REFERENCE_MATCH_THRESHOLD = 0.91

def strip_person_title(name):
    text = basic_name_clean(name)
    title_pattern = (
        r"^(นาย|นาง|นางสาว|ดร|พญ|นพ|คุณหญิง|หม่อมหลวง|"
        r"พลเอก|พลโท|พลตรี|พลเรือเอก|พลเรือโท|พลเรือตรี|"
        r"พลอากาศเอก|พลอากาศโท|พลอากาศตรี|พลตำรวจเอก|ร้อยเอก|พันเอก|พันโท|พันตรี)\s*"
    )
    prev = None
    while prev != text:
        prev = text
        text = re.sub(title_pattern, "", text).strip()
    return text

def load_reference_names_from_csv(url):
    df = None
    last_error = None
    for enc in ["utf-8-sig", "utf-8", "cp874", "tis-620"]:
        try:
            df = pd.read_csv(url, encoding=enc)
            break
        except Exception as ex:
            last_error = ex
    if df is None:
        raise RuntimeError(f"Failed to read {url}: {last_error}")

    df.columns = [str(c).strip() for c in df.columns]
    cols = list(df.columns)

    first_col = next((c for c in cols if ("ชื่อ" in c and "สกุล" not in c and "คำนำ" not in c)), None)
    last_col = next((c for c in cols if "นามสกุล" in c), None)
    full_col = next((c for c in cols if ("ชื่อ" in c and "สกุล" in c)), None)

    names = []
    if first_col and last_col:
        full_series = (
            df[first_col].fillna("").astype(str).str.strip()
            + " " +
            df[last_col].fillna("").astype(str).str.strip()
        ).str.strip()
        names.extend(full_series.tolist())
    elif full_col:
        names.extend(df[full_col].dropna().astype(str).tolist())
    else:
        # Fallback: gather probable name columns
        for c in cols:
            if "ชื่อ" in c or "name" in c.lower():
                names.extend(df[c].dropna().astype(str).tolist())

    cleaned = []
    for n in names:
        name_clean = strip_person_title(n)
        if len(name_clean) >= 6 and re.search(r"[\u0E00-\u0E7F]", name_clean):
            cleaned.append(name_clean)

    return sorted(set(cleaned))

def best_reference_match(name, reference_names, threshold=0.91):
    source = strip_person_title(name)
    source_key = re.sub(r"\s+", "", source)
    if len(source_key) < 6 or not reference_names:
        return source, 0.0

    best_name = source
    best_score = 0.0
    for ref in reference_names:
        ref_key = re.sub(r"\s+", "", strip_person_title(ref))
        score = SequenceMatcher(None, source_key, ref_key).ratio()
        if score > best_score:
            best_score = score
            best_name = ref

    if best_score >= threshold:
        return best_name, best_score
    return source, best_score

senate_reference_names = []
reference_load_errors = []
for ref_url in SENATE_REFERENCE_CSV_URLS:
    try:
        senate_reference_names.extend(load_reference_names_from_csv(ref_url))
    except Exception as ex:
        reference_load_errors.append(str(ex))
senate_reference_names = sorted(set(senate_reference_names))

senate_mask = all_votes_long["voter_party_norm"] == "สมาชิกวุฒิสภา"
senate_unique_names = sorted(all_votes_long.loc[senate_mask, "voter_name_norm"].dropna().astype(str).unique())

senate_reference_map = {}
senate_reference_score = {}
for name in senate_unique_names:
    mapped_name, score = best_reference_match(name, senate_reference_names, threshold=REFERENCE_MATCH_THRESHOLD)
    senate_reference_map[name] = mapped_name
    senate_reference_score[name] = score

all_votes_long.loc[senate_mask, "voter_name_norm"] = (
    all_votes_long.loc[senate_mask, "voter_name_norm"].map(senate_reference_map)
 )

# Recompute final tables after reference mapping
person_absence = (
    all_votes_long
    .groupby(["voter_party_norm", "voter_name_norm"], as_index=False)[["absent", "present"]]
    .sum()
)
person_absence["absence_rate"] = (
    person_absence["absent"] / (person_absence["absent"] + person_absence["present"])
)

party_average_absence = (
    person_absence.groupby("voter_party_norm", as_index=False)
    .agg(
        people=("voter_name_norm", "nunique"),
        avg_absence_rate=("absence_rate", "mean")
    )
    .rename(columns={"voter_party_norm": "voter_party"})
    .sort_values("avg_absence_rate", ascending=False)
)
party_average_absence["avg_absence_pct"] = (party_average_absence["avg_absence_rate"] * 100).round(2)

name_mapping_review = (
    all_votes_long[["voter_party_norm", "voter_name_raw", "voter_name_clean", "voter_name_norm"]]
    .drop_duplicates()
    .sort_values(["voter_party_norm", "voter_name_norm", "voter_name_clean"])
)

senate_reference_review = (
    pd.DataFrame({
        "name_before_reference": list(senate_reference_map.keys()),
        "name_after_reference": list(senate_reference_map.values()),
        "match_score": [senate_reference_score[k] for k in senate_reference_map.keys()],
    })
    .sort_values(["name_after_reference", "match_score"], ascending=[True, False])
)
senate_reference_review = senate_reference_review[
    senate_reference_review["name_before_reference"] != senate_reference_review["name_after_reference"]
]

print(f"Loaded senate reference names: {len(senate_reference_names)}")
if reference_load_errors:
    print("Reference load warnings:")
    for msg in reference_load_errors:
        print("-", msg)

senate_reference_review.head(50)

Loaded senate reference names: 486


,name_before_reference,name_after_reference,match_score
5,กฤษณุเหลืองพิบูลกิจ,กฤษณุ เหลืองพิบูลกิจ,1.000000
6,กลขัย สุวรรณบูรณ์,กลชัย สุวรรณบูรณ์,0.937500
26,กิตติพันธ์อนันตกูลจิรโชติ,กิตติพันธ์ อนันตกูลจิรโชติ,1.000000
53,จเรศักถิ์ อานุภาพ,จเรศักณิ์ อานุภาพ,0.937500
56,ฉัตรขัย สาริกัลยะ,ฉัตรชัย สาริกัลยะ,0.937500
212,พลเรือเอก ชัยวัฒน์ เอี่ยมสมุทร,ชัยวัฒน์ เอี่ยมสมุทร,1.000000
74,ชินโชติแสงสังข์,ชินโชติ แสงสังข์,1.000000
222,พลเรือเอกชุมนุม อาจวงษ์,ชุมนุม อาจวงษ์,1.000000
78,ชูชาติอินสว่าง,ชูชาติ อินสว่าง,1.000000
213,พลเรือเอก ฐนิธ กิตติอำพน,ฐนิธ กิตติอำพน,1.000000


In [123]:
# Review OCR merges for party and person names
from IPython.display import display

party_ocr_fix = party_mapping_review[
    party_mapping_review["voter_party_raw"] != party_mapping_review["voter_party_norm"]
]

name_ocr_fix = name_mapping_review[
    name_mapping_review["voter_name_clean"] != name_mapping_review["voter_name_norm"]
]

name_count_compare = (
    all_votes_long.groupby("voter_party_norm", as_index=False)
    .agg(
        raw_name_count=("voter_name_clean", "nunique"),
        norm_name_count=("voter_name_norm", "nunique"),
    )
)
name_count_compare["merged_names"] = (
    name_count_compare["raw_name_count"] - name_count_compare["norm_name_count"]
)
name_count_compare = name_count_compare.sort_values("merged_names", ascending=False)

display(name_count_compare.head(20))
display(party_ocr_fix)
display(name_ocr_fix.head(300))

,voter_party_norm,raw_name_count,norm_name_count,merged_names
26,สมาชิกวุฒิสภา,957,491,466
30,เพื่อไทย,498,298,200
1,ก้าวไกล,315,198,117
16,พลังประชารัฐ,311,202,109
6,ประชาชน,249,145,104
21,ภูมิใจไทย,228,140,88
9,ประชาธิปัตย์,157,97,60
24,รวมไทยสร้างชาติ,72,42,30
7,ประชาชาติ,50,25,25
0,กล้าธรรม,44,26,18


,voter_party_raw,voter_party_norm
0,..,
2,_,
3,t.,
4,กลาธรรม ู,กล้าธรรม
5,กลาธรรม 0,กล้าธรรม
20,พรรคกลาธรรม^๖,กล้าธรรม
21,พรรคกลาธรรมคb,กล้าธรรม
10,ชาติพัฒนากล้า,ชาติพัฒนา
22,"พรรคชาติพัฒนา j""",ชาติพัฒนา
23,พรรคชาติพัฒนาู๖,ชาติพัฒนา


,voter_party_norm,voter_name_raw,voter_name_clean,voter_name_norm
51735,กล้าธรรม,ก้องเกียรติ-เกตุสมบัติ,ก้องเกียรติเกตุสมบัติ,ก้องเกียรติ เกตุสมบัติ
27639,กล้าธรรม,จตุพร-กมลพันธ์ทิพย์,จตุพรกมลพันธ์ทิพย์,จตุพร กมลพันธ์ทิพย์
31581,กล้าธรรม,ชนนพัฒฐ์-นาคสั้ว,ชนนพัฒฐ์นาคสั้ว,ชนนพัฒฐ์ นาคสั้ว
32567,กล้าธรรม,ชัยทิพย์-กมลพันธ์ทิพย์,ชัยทิพย์กมลพันธ์ทิพย์,ชัยทิพย์ กมลพันธ์ทิพย์
27646,กล้าธรรม,ธรรมนัส-พรหมเผ่า,ธรรมนัสพรหมเผ่า,ธรรมนัส พรหมเผ่า
...,...,...,...,...
1813,ประชาธิปัตย์,จุรินทร์ ลักษณ์วิศิษฎ์,จุรินทร์ ลักษณ์วิศิษฎ์,จุรินทร์ ลักษณ์วิศิษ์ฎ์
7529,ประชาธิปัตย์,ชวน-หลีกภัย,ชวนหลีกภัย,ชวน หลีกภัย
5369,ประชาธิปัตย์,ชัยชนะ-เดชเดโช,ชัยชนะเดชเดโช,ชัยชนะ เดชเดโช
84165,ประชาธิปัตย์,ขัยทิพย์ กมลพันธ์ทิพย์,ขัยทิพย์ กมลพันธ์ทิพย์,ชัยทิพย์ กมลพันธ์ทิพย์


In [124]:
# Name checks: list + nunique
required_cols = {"voter_name_raw", "voter_name_clean", "voter_name_norm", "voter_party_norm"}
missing_cols = required_cols - set(all_votes_long.columns)
if missing_cols:
    raise ValueError(f"Missing columns {missing_cols}. Run Cell 5 first.")

name_check = all_votes_long[["voter_party_norm", "voter_name_raw", "voter_name_clean", "voter_name_norm"]].copy()

# Overall nunique summary
name_nunique_summary = pd.DataFrame(
    {
        "name_type": [
            "raw (voter_name_raw)",
            "clean (voter_name_clean)",
            "normalized (voter_name_norm)",
        ],
        "nunique": [
            name_check["voter_name_raw"].nunique(),
            name_check["voter_name_clean"].nunique(),
            name_check["voter_name_norm"].nunique(),
        ],
    }
)

# Per-party nunique summary
name_nunique_by_party = (
    name_check.groupby("voter_party_norm", as_index=False)
    .agg(
        raw_nunique=("voter_name_clean", "nunique"),
        norm_nunique=("voter_name_norm", "nunique"),
    )
)
name_nunique_by_party["merged_names"] = (
    name_nunique_by_party["raw_nunique"] - name_nunique_by_party["norm_nunique"]
)
name_nunique_by_party = name_nunique_by_party.sort_values("merged_names", ascending=False)

# Full normalized name list
names_list = (
    name_check[["voter_name_norm"]]
    .drop_duplicates()
    .sort_values("voter_name_norm")
    .reset_index(drop=True)
)

display(name_nunique_summary)
display(name_nunique_by_party)
display(names_list)

,name_type,nunique
0,raw (voter_name_raw),2800
1,clean (voter_name_clean),2777
2,normalized (voter_name_norm),1585


,voter_party_norm,raw_nunique,norm_nunique,merged_names
26,สมาชิกวุฒิสภา,957,491,466
30,เพื่อไทย,498,298,200
1,ก้าวไกล,315,198,117
16,พลังประชารัฐ,311,202,109
6,ประชาชน,249,145,104
21,ภูมิใจไทย,228,140,88
9,ประชาธิปัตย์,157,97,60
24,รวมไทยสร้างชาติ,72,42,30
7,ประชาชาติ,50,25,25
0,กล้าธรรม,44,26,18


,voter_name_norm
0,กนก ลิ้มตระกูล
1,กนก วงษ์ตระหง่าน
2,กนิษฐ์ ชาญปรีชญา
3,กมนทรรศน์ กิตติสุนทรสกุล
4,กมล รอดคล้าย
...,...
1580,ไพบูลย์ นิติตะวัน
1581,ไพลิน เทียนสุวรรณ
1582,ไพโรจน์ พานิชสมัย
1583,ไพโรจน์ พ่วงทอง


In [126]:
# Average entropy of each party by vote event (exclude absent votes)
import numpy as np

def normalize_votes_with_event(votes_df):
    if votes_df.empty:
        return pd.DataFrame(columns=["event_id", "voter_name_raw", "voter_party", "option"])

    out = votes_df.reindex(columns=["event_id", "voter_name_raw", "voter_party", "option"]).copy()
    out["event_id"] = out["event_id"].astype(str).str.strip()
    out["voter_name_raw"] = out["voter_name_raw"].astype(str).str.strip()
    out["voter_party"] = out["voter_party"].astype(str).str.strip()
    out["option"] = out["option"].astype(str).str.strip()
    out = out[
        (out["event_id"] != "")
        & (out["voter_party"] != "")
        & (out["voter_party"] != "-")
        & (out["voter_party"].str.lower() != "nan")
        & (out["option"] != "")
    ].copy()
    return out

def extract_vev_event_votes(vev_df):
    rows = []
    for row_idx, raw_votes in vev_df["votes"].items():
        event_id = f"vev_{row_idx}"
        for vote in parse_json_array(raw_votes):
            if isinstance(vote, dict):
                rows.append(
                    {
                        "event_id": event_id,
                        "voter_name_raw": vote.get("voter_name_raw", ""),
                        "voter_party": vote.get("voter_party", ""),
                        "option": vote.get("option", ""),
                    }
                )
    return normalize_votes_with_event(pd.DataFrame(rows))

def extract_bvev_event_votes(bvev_df):
    rows = []
    for row_idx, raw_events in bvev_df["vote_events"].items():
        events = parse_json_array(raw_events)
        for event_idx, event in enumerate(events):
            if not isinstance(event, dict):
                continue

            event_id = f"bvev_{row_idx}_{event_idx}"
            event_votes = event.get("votes", [])
            if not isinstance(event_votes, list):
                continue

            for vote in event_votes:
                if isinstance(vote, dict):
                    rows.append(
                        {
                            "event_id": event_id,
                            "voter_name_raw": vote.get("voter_name_raw", ""),
                            "voter_party": vote.get("voter_party", ""),
                            "option": vote.get("option", ""),
                        }
                    )
    return normalize_votes_with_event(pd.DataFrame(rows))

vev_event_votes = extract_vev_event_votes(vev)
bvev_event_votes = extract_bvev_event_votes(bvev)
entropy_votes_long = pd.concat([vev_event_votes, bvev_event_votes], ignore_index=True)

entropy_party_choices = party_choices if "party_choices" in globals() else sorted(set(CANONICAL_PARTIES))
entropy_votes_long["voter_party_norm"] = entropy_votes_long["voter_party"].apply(
    lambda x: normalize_party_name(x, entropy_party_choices)
)
entropy_votes_long = entropy_votes_long[
    (entropy_votes_long["voter_party_norm"] != "")
    & (entropy_votes_long["voter_party_norm"] != "-")
]

# Do not count absent votes when computing entropy.
present_entropy_votes = entropy_votes_long[entropy_votes_long["option"] != ABSENT_OPTION].copy()

if present_entropy_votes.empty:
    party_event_entropy = pd.DataFrame(
        columns=["event_id", "voter_party_norm", "party_present_votes", "distinct_options", "entropy_bits"]
    )
    party_average_entropy = pd.DataFrame(
        columns=["voter_party", "vote_events", "avg_present_votes", "avg_distinct_options", "avg_entropy_bits"]
    )
else:
    party_option_counts = (
        present_entropy_votes.groupby(["event_id", "voter_party_norm", "option"], as_index=False)
        .size()
        .rename(columns={"size": "vote_count"})
    )

    party_event_totals = (
        party_option_counts.groupby(["event_id", "voter_party_norm"], as_index=False)["vote_count"]
        .sum()
        .rename(columns={"vote_count": "party_present_votes"})
    )

    party_option_dist = party_option_counts.merge(
        party_event_totals,
        on=["event_id", "voter_party_norm"],
        how="left",
    )
    party_option_dist["p"] = party_option_dist["vote_count"] / party_option_dist["party_present_votes"]
    party_option_dist["entropy_term"] = -party_option_dist["p"] * np.log2(party_option_dist["p"])

    party_event_entropy = (
        party_option_dist.groupby(["event_id", "voter_party_norm"], as_index=False)
        .agg(
            party_present_votes=("party_present_votes", "first"),
            distinct_options=("option", "nunique"),
            entropy_bits=("entropy_term", "sum"),
        )
    )

    party_average_entropy = (
        party_event_entropy.groupby("voter_party_norm", as_index=False)
        .agg(
            vote_events=("event_id", "nunique"),
            avg_present_votes=("party_present_votes", "mean"),
            avg_distinct_options=("distinct_options", "mean"),
            avg_entropy_bits=("entropy_bits", "mean"),
        )
        .rename(columns={"voter_party_norm": "voter_party"})
        .sort_values("avg_entropy_bits", ascending=False)
        .reset_index(drop=True)
    )
    party_average_entropy["avg_present_votes"] = party_average_entropy["avg_present_votes"].round(2)
    party_average_entropy["avg_distinct_options"] = party_average_entropy["avg_distinct_options"].round(2)
    party_average_entropy["avg_entropy_bits"] = party_average_entropy["avg_entropy_bits"].round(4)

party_average_entropy

,voter_party,vote_events,avg_present_votes,avg_distinct_options,avg_entropy_bits
0,สมาชิกวุฒิสภา,84,175.49,2.82,0.3562
1,ประชาชาติ,357,6.03,1.34,0.2555
2,เศรษฐกิจไทย,77,9.60,1.35,0.2260
3,ก้าวไกล,212,47.22,1.54,0.1870
4,เพื่อชาติ,166,2.84,1.23,0.1810
5,เพื่อไทย,373,85.61,1.76,0.1505
6,ประชาธิปัตย์,373,26.13,1.57,0.1424
7,เศรษฐกิจใหม่,187,4.18,1.17,0.1330
8,ไทยสร้างไทย,186,4.48,1.15,0.1284
9,เสรีรวมไทย,220,4.24,1.17,0.1264


In [ ]:
party_average_entropy.to_csv('entropy.csv')

In [128]:
# Quick preview for party average entropy
party_average_entropy.head(30)

,voter_party,vote_events,avg_present_votes,avg_distinct_options,avg_entropy_bits
0,สมาชิกวุฒิสภา,84,175.49,2.82,0.3562
1,ประชาชาติ,357,6.03,1.34,0.2555
2,เศรษฐกิจไทย,77,9.60,1.35,0.2260
3,ก้าวไกล,212,47.22,1.54,0.1870
4,เพื่อชาติ,166,2.84,1.23,0.1810
5,เพื่อไทย,373,85.61,1.76,0.1505
6,ประชาธิปัตย์,373,26.13,1.57,0.1424
7,เศรษฐกิจใหม่,187,4.18,1.17,0.1330
8,ไทยสร้างไทย,186,4.48,1.15,0.1284
9,เสรีรวมไทย,220,4.24,1.17,0.1264


In [129]:
# Plain-text summary for entropy results
summary_cols = ["voter_party", "vote_events", "avg_present_votes", "avg_distinct_options", "avg_entropy_bits"]
print("party_average_entropy rows:", len(party_average_entropy))
print(party_average_entropy[summary_cols].head(20).to_string(index=False))

party_average_entropy rows: 41
         voter_party  vote_events  avg_present_votes  avg_distinct_options  avg_entropy_bits
       สมาชิกวุฒิสภา           84             175.49                  2.82            0.3562
           ประชาชาติ          357               6.03                  1.34            0.2555
         เศรษฐกิจไทย           77               9.60                  1.35            0.2260
             ก้าวไกล          212              47.22                  1.54            0.1870
           เพื่อชาติ          166               2.84                  1.23            0.1810
            เพื่อไทย          373              85.61                  1.76            0.1505
        ประชาธิปัตย์          373              26.13                  1.57            0.1424
        เศรษฐกิจใหม่          187               4.18                  1.17            0.1330
         ไทยสร้างไทย          186               4.48                  1.15            0.1284
          เสรีรวมไทย          220      

In [130]:
import pandas as pd 
import numpy as np 

o_vev = pd.read_csv('new_vev.csv')
o_bvev = pd.read_csv('new_bvev.csv')
new_vev = o_vev.iloc[1066:1334].reset_index(drop=True)

new_bvev = o_bvev.iloc[0:631].copy()
new_bvev.loc[new_bvev["vote_events"] == "[]", "vote_events"] = pd.NA
new_bvev = new_bvev.dropna(subset=["vote_events"]).reset_index(drop=True)

In [ ]:
# Party follow ratio vs government by vote date (new_vev + new_bvev)
GOV_BOUNDARY_1 = pd.Timestamp("2023-05-01")
GOV_BOUNDARY_2 = pd.Timestamp("2025-10-01")

def get_government_party(vote_date):
    if pd.isna(vote_date):
        return pd.NA
    vote_date = pd.Timestamp(vote_date)
    if vote_date < GOV_BOUNDARY_1:
        return "พลังประชารัฐ"
    if vote_date < GOV_BOUNDARY_2:
        return "เพื่อไทย"
    return "ภูมิใจไทย"

def normalize_votes_with_date(votes_df):
    if votes_df.empty:
        return pd.DataFrame(columns=["event_id", "end_date", "voter_name_raw", "voter_party", "option"])

    out = votes_df.reindex(columns=["event_id", "end_date", "voter_name_raw", "voter_party", "option"]).copy()
    out["event_id"] = out["event_id"].astype(str).str.strip()
    out["end_date"] = pd.to_datetime(out["end_date"], errors="coerce")
    out["voter_name_raw"] = out["voter_name_raw"].astype(str).str.strip()
    out["voter_party"] = out["voter_party"].astype(str).str.strip()
    out["option"] = out["option"].astype(str).str.strip()
    out = out[
        (out["event_id"] != "")
        & out["end_date"].notna()
        & (out["voter_party"] != "")
        & (out["voter_party"] != "-")
        & (out["voter_party"].str.lower() != "nan")
        & (out["option"] != "")
    ].copy()
    return out

def extract_new_vev_votes_with_date(df):
    rows = []
    for row_idx, row in df.iterrows():
        event_id = f"new_vev_{row_idx}"
        event_end_date = row.get("end_date", pd.NA)
        for vote in parse_json_array(row.get("votes", "[]")):
            if isinstance(vote, dict):
                rows.append(
                    {
                        "event_id": event_id,
                        "end_date": event_end_date,
                        "voter_name_raw": vote.get("voter_name_raw", ""),
                        "voter_party": vote.get("voter_party", ""),
                        "option": vote.get("option", ""),
                    }
                )
    return normalize_votes_with_date(pd.DataFrame(rows))

def extract_new_bvev_votes_with_date(df):
    rows = []
    for row_idx, raw_events in df["vote_events"].items():
        events = parse_json_array(raw_events)
        for event_idx, event in enumerate(events):
            if not isinstance(event, dict):
                continue

            event_id = f"new_bvev_{row_idx}_{event_idx}"
            event_end_date = event.get("end_date", pd.NA)
            event_votes = event.get("votes", [])
            if not isinstance(event_votes, list):
                continue

            for vote in event_votes:
                if isinstance(vote, dict):
                    rows.append(
                        {
                            "event_id": event_id,
                            "end_date": event_end_date,
                            "voter_name_raw": vote.get("voter_name_raw", ""),
                            "voter_party": vote.get("voter_party", ""),
                            "option": vote.get("option", ""),
                        }
                    )
    return normalize_votes_with_date(pd.DataFrame(rows))

new_vev_votes_with_date = extract_new_vev_votes_with_date(new_vev)
new_bvev_votes_with_date = extract_new_bvev_votes_with_date(new_bvev)
new_votes_long = pd.concat([new_vev_votes_with_date, new_bvev_votes_with_date], ignore_index=True)

new_party_choices = party_choices if "party_choices" in globals() else sorted(set(CANONICAL_PARTIES))
new_votes_long["voter_party_norm"] = new_votes_long["voter_party"].apply(
    lambda x: normalize_party_name(x, new_party_choices)
)
new_votes_long = new_votes_long[
    (new_votes_long["voter_party_norm"] != "")
    & (new_votes_long["voter_party_norm"] != "-")
]

new_votes_long["government_party"] = new_votes_long["end_date"].apply(get_government_party)

# Exclude absent when measuring follow ratio.
new_votes_present = new_votes_long[new_votes_long["option"] != ABSENT_OPTION].copy()

# Government position in each event = majority option of active government party.
gov_votes = new_votes_present[
    new_votes_present["voter_party_norm"] == new_votes_present["government_party"]
]

gov_option_counts = (
    gov_votes.groupby(["event_id", "option"], as_index=False)
    .size()
    .rename(columns={"size": "votes"})
)
gov_majority_option = (
    gov_option_counts.sort_values(["event_id", "votes", "option"], ascending=[True, False, True])
    .drop_duplicates(subset=["event_id"], keep="first")
    .rename(columns={"option": "government_option", "votes": "government_option_votes"})
    [["event_id", "government_option", "government_option_votes"]]
)

# Event-level party follow ratio: 1 if all match government option, 0 if none match.
event_follow_votes = new_votes_present.merge(gov_majority_option, on="event_id", how="inner")
event_follow_votes["follow_government"] = (
    event_follow_votes["option"] == event_follow_votes["government_option"]
).astype(float)

party_event_follow_ratio = (
    event_follow_votes.groupby(
        ["event_id", "end_date", "government_party", "government_option", "voter_party_norm"],
        as_index=False,
    )
    .agg(
        party_present_votes=("follow_government", "size"),
        follow_ratio=("follow_government", "mean"),
    )
    .rename(columns={"voter_party_norm": "voter_party"})
)
party_event_follow_ratio["is_government_party"] = (
    party_event_follow_ratio["voter_party"] == party_event_follow_ratio["government_party"]
)
party_event_follow_ratio["party_role"] = np.where(
    party_event_follow_ratio["is_government_party"], "government", "opposition"
)

# Average follow ratio over all events where the party appears (both roles mixed).
party_average_follow_ratio = (
    party_event_follow_ratio.groupby("voter_party", as_index=False)
    .agg(
        vote_events=("event_id", "nunique"),
        avg_follow_ratio=("follow_ratio", "mean"),
        avg_party_present_votes=("party_present_votes", "mean"),
    )
    .sort_values("avg_follow_ratio", ascending=False)
    .reset_index(drop=True)
)
party_average_follow_ratio["avg_follow_pct"] = (party_average_follow_ratio["avg_follow_ratio"] * 100).round(2)
party_average_follow_ratio["avg_party_present_votes"] = party_average_follow_ratio["avg_party_present_votes"].round(2)

# Split by role to avoid mixing periods where a party is government vs opposition.
party_role_average_follow_ratio = (
    party_event_follow_ratio.groupby(["voter_party", "party_role"], as_index=False)
    .agg(
        vote_events=("event_id", "nunique"),
        avg_follow_ratio=("follow_ratio", "mean"),
    )
    .sort_values(["party_role", "avg_follow_ratio"], ascending=[True, False])
    .reset_index(drop=True)
)
party_role_average_follow_ratio["avg_follow_pct"] = (
    party_role_average_follow_ratio["avg_follow_ratio"] * 100
).round(2)

opposition_average_follow_ratio = (
    party_role_average_follow_ratio[party_role_average_follow_ratio["party_role"] == "opposition"]
    .drop(columns=["party_role"])
    .reset_index(drop=True)
)

government_average_follow_ratio = (
    party_role_average_follow_ratio[party_role_average_follow_ratio["party_role"] == "government"]
    .drop(columns=["party_role"])
    .reset_index(drop=True)
)

party_average_follow_ratio.to_csv("party_average_follow_ratio.csv", index=False)
party_event_follow_ratio.to_csv("party_event_follow_ratio.csv", index=False)
party_role_average_follow_ratio.to_csv("party_role_average_follow_ratio.csv", index=False)
opposition_average_follow_ratio.to_csv("opposition_average_follow_ratio.csv", index=False)
government_average_follow_ratio.to_csv("government_average_follow_ratio.csv", index=False)

party_average_follow_ratio

,voter_party,vote_events,avg_follow_ratio,avg_party_present_votes,avg_follow_pct
0,ประชานิยม,2,1.000000,1.00,100.00
1,พลังสังคมใหม่,35,1.000000,1.00,100.00
2,โอกาสไทย,2,1.000000,1.00,100.00
3,ไทยรวมไทย,15,1.000000,1.00,100.00
4,รวมแผ่นดิน,2,1.000000,1.00,100.00
5,พรร่คภูมีใจไทย,2,1.000000,1.00,100.00
6,พลังไทยรักไทย,6,1.000000,1.00,100.00
7,รวมพลัง,196,0.992347,2.01,99.23
8,ประชาภิวัฒน์,146,0.979452,1.00,97.95
9,ประชาธรรมไทย,76,0.973684,1.00,97.37


In [141]:
party_average_follow_ratio

,voter_party,vote_events,avg_follow_ratio,avg_party_present_votes,avg_follow_pct
0,ประชานิยม,2,1.000000,1.00,100.00
1,พลังสังคมใหม่,35,1.000000,1.00,100.00
2,โอกาสไทย,2,1.000000,1.00,100.00
3,ไทยรวมไทย,15,1.000000,1.00,100.00
4,รวมแผ่นดิน,2,1.000000,1.00,100.00
5,พรร่คภูมีใจไทย,2,1.000000,1.00,100.00
6,พลังไทยรักไทย,6,1.000000,1.00,100.00
7,รวมพลัง,196,0.992347,2.01,99.23
8,ประชาภิวัฒน์,146,0.979452,1.00,97.95
9,ประชาธรรมไทย,76,0.973684,1.00,97.37


In [138]:
# Plain-text summary: follow ratio against government
cols_all = ["voter_party", "vote_events", "avg_follow_ratio", "avg_follow_pct", "avg_party_present_votes"]
cols_role = ["voter_party", "vote_events", "avg_follow_ratio", "avg_follow_pct"]

print("party_average_follow_ratio rows:", len(party_average_follow_ratio))
print(party_average_follow_ratio[cols_all].head(20).to_string(index=False))

print("\nOpposition-only rows:", len(opposition_average_follow_ratio))
print(opposition_average_follow_ratio[cols_role].head(20).to_string(index=False))

print("\nGovernment-only rows:", len(government_average_follow_ratio))
print(government_average_follow_ratio[cols_role].head(20).to_string(index=False))

party_average_follow_ratio rows: 41
         voter_party  vote_events  avg_follow_ratio  avg_follow_pct  avg_party_present_votes
           ประชานิยม            2          1.000000          100.00                     1.00
       พลังสังคมใหม่           35          1.000000          100.00                     1.00
            โอกาสไทย            2          1.000000          100.00                     1.00
           ไทยรวมไทย           15          1.000000          100.00                     1.00
          รวมแผ่นดิน            2          1.000000          100.00                     1.00
      พรร่คภูมีใจไทย            2          1.000000          100.00                     1.00
       พลังไทยรักไทย            6          1.000000          100.00                     1.00
             รวมพลัง          196          0.992347           99.23                     2.01
        ประชาภิวัฒน์          146          0.979452           97.95                     1.00
        ประชาธรรมไทย           76 